# Étape 6 — Final Backtest & Mémoire Conclusion
## MASI Hybrid Forecasting System

Backtest of all 5 models on the 948-day TEST with **Deflated Sharpe Ratio** (Bailey & López de Prado 2014), **regime-conditional decomposition**, and **cost sensitivity** at 5/10/20 bps.

**Key findings — see `etape6_final_report.md` §0.**
1. CNN-LSTM has the best DA (0.556) AND best cost-robustness (still +0.42 Sharpe at 20 bps; ARIMA dies at 20 bps).
2. ALL models lose in the Neutral regime; CNN-LSTM is the best trader in Bear (+2.07) AND Bull (+2.50).
3. Honest DSR: no model crosses the 0.95 publishable threshold — best is ARIMA 0.62, CNN-LSTM 0.53.

## 1. Run the backtest

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath('.'))

import pandas as pd
from IPython.display import Image, display

import importlib.util as _ilu, sys as _sys; _spec = _ilu.spec_from_file_location('e6', '../scripts/06_backtesting.py'); e6 = _ilu.module_from_spec(_spec); _sys.modules['e6'] = e6; _spec.loader.exec_module(e6)
e6.main()

## 2. Per-model backtest summary (5 bps)

In [ ]:
bt = json.load(open(os.path.join(e6.RESULTS_DIR, 'backtest_metrics.json')))
rows = []
for m, d in bt['models'].items():
    rows.append({'model': e6.MODEL_LABELS[m], 'DA': round(d['directional_accuracy'], 4),
                 'Sharpe': round(d['sharpe'], 3), 'Sortino': round(d['sortino'], 3),
                 'MDD': round(d['max_drawdown'], 3), 'Calmar': round(d['calmar'], 2),
                 'final_equity': round(d['final_equity'], 3),
                 'DSR': round(d['deflated']['deflated_sharpe_psr'], 3)})
pd.DataFrame(rows).set_index('model')

## 3. Regime-conditional Sharpe (THE key insight)

In [ ]:
rows = []
for m, rt in bt['regime_conditional'].items():
    rows.append({'model': e6.MODEL_LABELS[m],
                 **{r: (round(rt[r]['sharpe'], 2) if rt[r]['sharpe'] is not None else None)
                    for r in e6.REGIME_NAMES}})
print('All models LOSE in the Neutral regime; CNN-LSTM wins both extremes:')
pd.DataFrame(rows).set_index('model')

## 4. Cost sensitivity

In [ ]:
rows = []
for cb, d in bt['cost_sensitivity'].items():
    rows.append({'cost_bps': int(cb), **{e6.MODEL_LABELS[m]: round(d[m], 2) for m in e6.MODELS}})
print('At 20 bps, ARIMA dies (-0.02); CNN-LSTM still profitable (+0.42):')
pd.DataFrame(rows).set_index('cost_bps')

## 5. Plots

In [ ]:
Image(os.path.join(e6.PLOTS_DIR, 'etape6_equity_curves.png'))

In [ ]:
Image(os.path.join(e6.PLOTS_DIR, 'etape6_drawdowns.png'))

In [ ]:
Image(os.path.join(e6.PLOTS_DIR, 'etape6_regime_heatmap.png'))

In [ ]:
Image(os.path.join(e6.PLOTS_DIR, 'etape6_cost_sensitivity.png'))

In [ ]:
Image(os.path.join(e6.PLOTS_DIR, 'etape6_final_summary.png'))

## 6. Final recommendation for the mémoire

| | Recommended model | Why |
|---|---|---|
| **Best directional prediction** | CNN-LSTM `base12` (DA 0.556) | wins DA, MDD, Calmar |
| **Most cost-robust trader** | CNN-LSTM `base12` | survives 20 bps; ARIMA dies |
| **Best in Bear regime** | CNN-LSTM (Sharpe +2.07) | beats ARIMA +1.51 |
| **Best in Bull regime** | CNN-LSTM (Sharpe +2.50) | beats ARIMA +1.75 |
| **Best aggregate Sharpe at 5 bps** | ARIMA (+1.17) | wins only this single metric, fragile |
| **Statistical robustness** | None (DSR < 0.95 for all) | honest, 948 days too few |

**Production model = CNN-LSTM `base12`** + regime-gating in future work.